# Performing Domain Analysis with AI  
## Leveraga SA Model, Write Up, and Results on Strategy to develop a first cut Architecture

First we load up the PEAK Model

### Model-Specific Code (Do Not Modify)

This section contains code that is specific to the system model. It is updated only when the model is changed and should not require user modifications under normal circumstances.

If a new model is introduced, ensure this section is reviewed and updated as needed.


In [1]:
#!pip install --upgrade git+https://github.com/tkSDISW/Capella_Tools 
import capellambse.decl

from capella_tools import capellambse_helper

from IPython import display as diag_display
resources = {
    "PEAK": "PEAK/PEAK",
}
path_to_model = "../PEAK.aird"
model = capellambse.MelodyModel(path_to_model, resources=resources)
from capella_tools import Pub4C
# Instantiate the class with the traceability file
traceability_store = Pub4C.Traceability_Store("../PEAK/traceability")


## 🔄 Embedding Generation Process

Next we load evaluate whether we ne to update the embedings. If there older than model we will update thems. 🚀


In [2]:

from capella_tools  import capella_embeddings_manager

# Generate embeddings for all objects
model_embedding_manager = capella_embeddings_manager.EmbeddingManager()

embedding_file = "embeddings.json" 
model_embedding_manager.set_files( path_to_model , embedding_file)

model_embedding_manager.create_model_embeddings(model)

OpenAI API Key retrieved successfully.
Creating Embeddings
embeddings generated
Saving embeddings


## 🎯 Define the model element or diagram for analysis.

We use the embedding to locate a diagram that will be used for basis of analysis. It a OCB diagrams with relations to all the functional chains. 

> 💡 **Tip:** If you're unsure about the model structure, review the documentation or refer to the model's diagrams for additional guidance.


In [3]:

selected_objects = model_embedding_manager.query_and_select_top_objects("[SAB] PEAK", top_n=1)



This is a list of ranked Objects Based on Query:
Index: 0, Name: PEAK, Similarity: 0.60, Type: SystemComponent, Phase: System Analysis SA, Source: , Target: 


## 📝 Generate Structured Input File 

We then generate the structured input file for all the matching objects and there related objects.
A file "capella_model.yaml" is written it you want to look at it. 

In [4]:
#Workflow
from capella_tools import capellambse_yaml_manager
yaml_handler = capellambse_yaml_manager.CapellaYAMLHandler()
   
#Generate YAML for the logical component and append to the file
for object in  selected_objects :  
    yaml_handler.generate_yaml(model.by_uuid(object["uuid"]))  

yaml_handler.generate_traceability_related_objects(model,traceability_store)

#yaml_handler.display()
yaml_handler.generate_yaml_referenced_objects()
#yaml_handler.display()

yaml_handler.write_output_file()


## 🖼️ Graph Analysis 

Not we generate a directed graph based on given relations.

In [5]:
import os
from openai import OpenAI
from IPython.core.display import HTML
from IPython.display import display, clear_output, Markdown, IFrame
from capella_tools  import Open_AI_RAG_manager


#print(object)
# Step 1: Get YAML content
yaml_content = yaml_handler.get_yaml_content()

# Step 2: Invoke ChatGPT for analysis
analyzer = Open_AI_RAG_manager.ChatGPTAnalyzer(yaml_content)

analyzer.analyze_and_generate_graph()


# Show inside the notebook
IFrame(src="graph.html", width="100%", height="750px")

OpenAI API Key retrieved successfully.
Sending prompt to extract graphable relations...


**Your prompt:** 
Analyze the YAML content and extract relationships suitable for a graph.

- Each relationship should be a tuple: (source, target, label).
- Use simple labels like 'abstract type of','property value group','property value','constrains','linked model element',"region','states','transitions','outgoing transition','incoming transtion','do function','entry function','exits function','triggers','source state','destination state','after function','source','target','involves','exchange items','exchanges','allocated to', 'involving','components','deployed to','port','state machine','entities', 'elements of'.
- Use ref_id to navigate primary_id but do not list them in tuples.
- Output ONLY a Python list of these tuples.
- No explanation, no extra text, just the list in Python syntax.
 Format the response in .html format.

**ChatGPT Response:**


[
    ("PEAK", "Provide Situational Awareness", "allocated to"),
    ("PEAK", "Provide Communication management", "allocated to"),
    ("PEAK", "Provide Power", "allocated to"),
    ("PEAK", "Provide Water", "allocated to"),
    ("PEAK", "Backup Power", "allocated to"),
    ("PEAK", "Receive Setup", "allocated to"),
    ("PEAK", "Build System", "allocated to"),
    ("PEAK", "Offer Kit", "allocated to"),
    ("PEAK", "Receive Build", "allocated to"),
    ("PEAK", "Receive Tranport", "allocated to"),
    ("PEAK", "Accept Storage", "allocated to"),
    ("PEAK", "System State Machine", "state machine"),
    ("Provide Situational Awareness", "inManager Situational Awareness", "port"),
    ("Provide Situational Awareness", "outComms", "port"),
    ("Provide Situational Awareness", "outRecon", "port"),
    ("Provide Situational Awareness", "outMonitor Environment", "port"),
    ("Provide Situational Awareness", "Operate System", "exchanges"),
    ("Provide Situational Awareness", "Provide Communication management", "exchanges"),
    ("Provide Situational Awareness", "Perform Aerial Recon", "exchanges"),
    ("Provide Situational Awareness", "Monitor Environment", "exchanges"),
    ("Provide Communication management", "inManage Communications", "port"),
    ("Provide Communication management", "inComms", "port"),
    ("Provide Communication management", "outCellular", "port"),
    ("Provide Communication management", "outComm Service", "port"),
    ("Provide Communication management", "outLocal Communciations", "port"),
    ("Provide Communication management", "outRegional National Comms", "port"),
    ("Provide Communication management", "Operate System", "exchanges"),
    ("Provide Communication management", "Provide Communncation Services", "exchanges"),
    ("Provide Communication management", "Provide National and Regial communications", "exchanges"),
    ("Provide Power", "inManage Power Generation", "port"),
    ("Provide Power", "inRenewable Power", "port"),
    ("Provide Power", "inPower", "port"),
    ("Provide Power", "outPower", "port"),
    ("Provide Power", "Operate System", "exchanges"),
    ("Provide Power", "Collect Energy", "exchanges"),
    ("Provide Power", "Backup Power", "exchanges"),
    ("Provide Power", "Receive Power for Phone", "exchanges"),
    ("Provide Water", "inManage Water Production", "port"),
    ("Provide Water", "outFiltered Water", "port"),
    ("Provide Water", "outWater", "port"),
    ("Provide Water", "Operate System", "exchanges"),
    ("Provide Water", "Receive Water", "exchanges"),
    ("Backup Power", "outPower", "port"),
    ("Backup Power", "Provide Power", "exchanges"),
    ("Receive Setup", "inSetup", "port"),
    ("Receive Setup", "Setup Kit", "exchanges"),
    ("Build System", "PEAK", "allocated to"),
    ("Offer Kit", "PEAK", "allocated to"),
    ("Receive Build", "inBuild", "port"),
    ("Receive Build", "Build Kit", "exchanges"),
    ("Receive Tranport", "inTranport", "port"),
    ("Receive Tranport", "Deliver the Kit", "exchanges"),
    ("Accept Storage", "inStore", "port"),
    ("Accept Storage", "Store Kit", "exchanges"),
    ("System State Machine", "PEAK Inital", "states"),
    ("System State Machine", "Building", "states"),
    ("System State Machine", "Inventoried", "states"),
    ("System State Machine", "Delivery", "states"),
    ("System State Machine", "Storing", "states"),
    ("System State Machine", "Deploying", "states"),
    ("System State Machine", "Deployed", "states"),
    ("System State Machine", "Retired", "states"),
    ("System State Machine", "", "transitions"),
    ("PEAK Inital", "", "outgoing transition"),
    ("Building", "Receive Build", "do function"),
    ("Inventoried", "", "outgoing transition"),
    ("Delivery", "Receive Tranport", "do function"),
    ("Storing", "", "outgoing transition"),
    ("Storing", "", "outgoing transition"),
    ("Deploying", "Receive Setup", "do function"),
    ("Deployed", "Backup Power", "do function"),
    ("Deployed", "Provide Power", "do function"),
    ("Deployed", "Provide Water", "do function"),
    ("Deployed", "Provide Situational Awareness", "do function"),
    ("Deployed", "Provide Communication management", "do function"),
    ("Retired", "", "incoming transition"),
    ("Retired", "", "incoming transition")
]


**Token Usage Info:**

Tokens used: prompt=16030, completion=1043, total=17073

Relations Text: 
[
    ("PEAK", "Provide Situational Awareness", "allocated to"),
    ("PEAK", "Provide Communication management", "allocated to"),
    ("PEAK", "Provide Power", "allocated to"),
    ("PEAK", "Provide Water", "allocated to"),
    ("PEAK", "Backup Power", "allocated to"),
    ("PEAK", "Receive Setup", "allocated to"),
    ("PEAK", "Build System", "allocated to"),
    ("PEAK", "Offer Kit", "allocated to"),
    ("PEAK", "Receive Build", "allocated to"),
    ("PEAK", "Receive Tranport", "allocated to"),
    ("PEAK", "Accept Storage", "allocated to"),
    ("PEAK", "System State Machine", "state machine"),
    ("Provide Situational Awareness", "inManager Situational Awareness", "port"),
    ("Provide Situational Awareness", "outComms", "port"),
    ("Provide Situational Awareness", "outRecon", "port"),
    ("Provide Situational Awareness", "outMonitor Environment", "port"),
    ("Provide Situational Awareness", "Operate System", "exchanges"),
    ("Provide Situational Awarene

## ⚙️ Execute  Prompt
Now we load up the PEAK System document. 

Then execute the prompt that will use the model and system document. 

In [7]:
analyzer.add_text_file_to_messages("PEAK System.docx")
prompt = """
Decompose the PEAK system into its logical architecture using the terms of the ARCADIA method, use the provide model and document as resource and use Use general domain knowledge to supplement the supplied content. 
Generate a hiearchical list of components and sub-components for system. 
-Provide a rationale for each component and sub-component.
-Note in rationale 
--the specific the source of any external domain knowledge used.
--Note in rational any requirement relevent to the component of sub-component.
"""
analyzer.follow_up_prompt(prompt)
chatgpt_response = analyzer.get_response()

✅ File `PEAK System.docx` added to messages for analysis.


**Your prompt:** 
Decompose the PEAK system into its logical architecture using the terms of the ARCADIA method, use the provide model and document as resource and use Use general domain knowledge to supplement the supplied content. 
Generate a hiearchical list of components and sub-components for system. 
-Provide a rationale for each component and sub-component.
-Note in rationale 
--the specific the source of any external domain knowledge used.
--Note in rational any requirement relevent to the component of sub-component.


**ChatGPT Response:**

### Logical Architecture of the PEAK System

1. **PEAK System**
   - **Rationale**: The PEAK System acts as a comprehensive kit designed for rapid deployment in crisis situations, providing critical services to enhance stability and security. The primary source is the provided document and model, supplemented by the general understanding of emergency response systems.

2. **Water Sub-System**
   - **Components**:
     - **Water Purification Equipment**
       - **Rationale**: Includes filtration, distribution capability, and storage container to provide potable water, an essential service during crises. Requirement: Capable of purifying fresh, brackish, or salt water, integrated with power generation (source: provided document).
     - **Filtration System**
       - **Rationale**: Ensures water is safe for drinking and hygiene. Requirement: Part of the purification process (source: domain knowledge of water safety standards).
     - **Storage Container**
       - **Rationale**: Provides a means to hold and distribute purified water. Requirement: Must meet durability and lightweight standards (source: logistical standards for emergency supplies).

3. **Power Sub-System**
   - **Components**:
     - **Renewable Power Sources**
       - **Rationale**: Utilizes solar, wind, etc., to generate sustainable energy with minimal environmental impact. Requirement: Must power all kit components (source: sustainable energy practices).
     - **Fossil Fuel Generator Backup**
       - **Rationale**: Provides reliability in diverse conditions where renewable sources may be insufficient. Requirement: Support system for power failure scenarios (source: best practices in power system reliability).

4. **Communication Sub-System**
   - **Components**:
     - **Communication Device**
       - **Rationale**: Enables voice and data communication over low-bandwidth networks. Requirement: Integration with situational awareness component (source: telecommunications standards).
     - **Distributed Communication Network**
       - **Rationale**: Facilitates mobile and decentralized communications, crucial for dynamic crisis environments (source: general knowledge on communication networks).

5. **Situational Awareness Sub-System**
   - **Components**:
     - **Unmanned Aerial System (UAS)**
       - **Rationale**: Provides an aerial perspective to assess environments and infrastructure quickly. Requirement: Integrates with power system (source: modern surveillance and reconnaissance technologies).
     - **Control Device and Platform**
       - **Rationale**: Manages UAS operations and data collection. Requirement: User-friendly for rapid deployment (source: user experience design in technology interfaces).

6. **System Integration**
   - **Rationale**: Ensures all sub-systems work cohesively to provide rapid, effective response in crisis situations. Requirement: Coordination of power, communication, and situational awareness (source: system engineering principles).

7. **Transportation and Deployment Sub-System**
   - **Components**:
     - **Cargo Transportation**
       - **Rationale**: The system must fit on a 463L cargo air pallet for easy transport via military and civilian modes. Requirement: Lightweight and transportable (source: logistical considerations for rapid deployment).
     - **Pre-positioned Kits**
       - **Rationale**: Quick deployment capability is ensured by pre-staging kits at strategic locations. Requirement: Efficient distribution in critical times (source: emergency logistics practices).

### Conclusion

Each component and sub-component of the PEAK System is critical for an efficient and effective response to crises, designed with considerations for sustainability, interoperability, and rapid deployment based on the provided PEAK System document, the model, and supplemented with general domain knowledge where necessary.

**Token Usage Info:**

Tokens used: prompt=18168, completion=735, total=18903

## 💬 Launch Interactive Chat on Structured Input




In [ ]:
analyzer.interactive_chat()


In [ ]:
print("Done")

# 